# Quality Checks: Silver tables
`Rerunning the queries again to check if there is no issues after the table is created`

**Expectation: No result or the standardized values**

## Table: silver.crm.customer_info

In [ ]:
%%sql
-- check for NULLS or duplicates in primary key column (cst_id)

SELECT cst_id, COUNT(*) AS id_count FROM sales_lakehouse.dbo.silver_crm_customer_info
GROUP BY cst_id
HAVING COUNT(*) > 1 OR cst_id IS NULL;

-- Observation: No results

In [ ]:
-- check for unwanted spaces in  cst_firstname
SELECT 
cst_firstname
FROM sales_lakehouse.dbo.silver_crm_customer_info
WHERE cst_firstname != TRIM(cst_firstname);

In [ ]:
-- check for unwanted spaces in cst_lastname
SELECT 
cst_lastname
FROM sales_lakehouse.dbo.silver_crm_customer_info
WHERE cst_lastname != TRIM(cst_lastname);

In [ ]:
-- cst_marital_status
SELECT DISTINCT cst_marital_status
FROM sales_lakehouse.dbo.silver_crm_customer_info;

In [ ]:
-- cst_gndr
SELECT DISTINCT cst_gndr
FROM sales_lakehouse.dbo.silver_crm_customer_info;

In [ ]:
-- check if all the rows have been loaded from bronze to silver
SELECT COUNT(*) FROM sales_lakehouse.dbo.bronze_crm_customer_info;
SELECT COUNT(*) FROM sales_lakehouse.dbo.silver_crm_customer_info;

-- Observation: there is a difference here, because we have removed duplicate primary keys 

## Table: silver.crm.product_info

In [ ]:
-- check for NULLS or duplicates in primary key column (prd_id)
SELECT prd_id, COUNT(*) AS id_count FROM sales_lakehouse.dbo.silver_crm_product_info
GROUP BY prd_id
HAVING COUNT(*) > 1 OR prd_id IS NULL;

In [ ]:
-- check for unwanted spaces in string columns

SELECT
prd_nm
FROM sales_lakehouse.dbo.silver_crm_product_info
WHERE prd_nm != TRIM(prd_nm);

In [ ]:
-- check for NULLS or Negative Numbers
SELECT
prd_cost
FROM sales_lakehouse.dbo.silver_crm_product_info
WHERE prd_cost < 0 OR prd_cost IS NULL;

In [ ]:
-- check issues in low cardinality columns
-- Data Standardization & Consistency
SELECT
DISTINCT prd_line
FROM sales_lakehouse.dbo.silver_crm_product_info;

In [ ]:
-- check for invalid date orders
-- End date must not be smaller than the start date
SELECT
*
FROM sales_lakehouse.dbo.silver_crm_product_info
WHERE prd_end_dt < prd_start_dt;

In [ ]:
-- check if all the rows have been loaded from bronze to silver
SELECT COUNT(*) FROM sales_lakehouse.dbo.bronze_crm_product_info;
SELECT COUNT(*) FROM sales_lakehouse.dbo.silver_crm_product_info;

## Table: silver_crm_sales_details

In [ ]:
-- check for invalid date orders
-- Order Date must always be earlier than the Shipping Date or Due Date
SELECT
*             -- 
FROM sales_lakehouse.dbo.silver_crm_sales_details
WHERE sls_order_dt > sls_ship_dt OR sls_order_dt > sls_due_dt;

-- observation: no results (expected)

In [ ]:
-- Check Data Consistency between Sales, Quantity and Price
--Business Rule:
--Sales = Quantity * Price
--Negative, zeros and NULLs are NOT Allowed in neither in Sales, Quantity and Price

SELECT
sls_sales,
sls_quantity,
sls_price
FROM sales_lakehouse.dbo.silver_crm_sales_details
WHERE sls_sales != sls_quantity * sls_price
OR sls_sales IS NULL OR sls_quantity IS NULL OR sls_price IS NULL
OR sls_sales <= 0 OR sls_quantity <= 0 OR sls_price <= 0
ORDER BY sls_sales, sls_quantity, sls_price;

-- observation: no results (expected)

In [ ]:
-- check if all the rows have been loaded from bronze to silver

SELECT COUNT(*) FROM sales_lakehouse.dbo.bronze_crm_sales_details;

SELECT COUNT(*) FROM sales_lakehouse.dbo.silver_crm_sales_details;

## Table: silver_erp_customers

In [ ]:
-- check if all cid is there in silver_crm_customer_info after transforming bronze_erp_customers
SELECT
cid
FROM sales_lakehouse.dbo.silver_erp_customers 
WHERE cid NOT IN
(SELECT DISTINCT cst_key FROM sales_lakehouse.dbo.silver_crm_customer_info);

-- observation: no result (expected)

In [ ]:
-- check if we have future birth dates in silver_erp_customers
SELECT
bdate
FROM sales_lakehouse.dbo.silver_erp_customers
WHERE bdate > CURRENT_TIMESTAMP();

-- observation: no result (expected)

In [ ]:
-- check data standardization in low cardinality column
SELECT
DISTINCT gen
FROM sales_lakehouse.dbo.silver_erp_customers;

In [ ]:
-- check if all the rows have been loaded from bronze to silver

SELECT COUNT(*) FROM sales_lakehouse.dbo.bronze_erp_customers;

SELECT COUNT(*) FROM sales_lakehouse.dbo.silver_erp_customers;

-- observation: counts are matching (expected)

## Table: silver_erp_location

In [ ]:
-- check if the all the cids are matching after transformation
SELECT 
cid
FROM sales_lakehouse.dbo.silver_erp_location
WHERE cid NOT IN
(SELECT cst_key FROM sales_lakehouse.dbo.silver_crm_customer_info);

-- observation: no result (expected)

In [ ]:
-- check if the country names are standardized in the silver table
SELECT
DISTINCT cntry
FROM sales_lakehouse.dbo.silver_erp_location;

In [ ]:
-- check if all the rows have been loaded from bronze to silver

SELECT COUNT(*) FROM sales_lakehouse.dbo.bronze_erp_location;

SELECT COUNT(*) FROM sales_lakehouse.dbo.silver_erp_location;

-- observation: counts are matching (expected)

## Table: silver_erp_product_category

In [ ]:
-- check if all the rows have been loaded from bronze to silver

SELECT COUNT(*) FROM sales_lakehouse.dbo.bronze_erp_product_category;

SELECT COUNT(*) FROM sales_lakehouse.dbo.silver_erp_product_category;

-- observation: counts are matching (expected)